<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/Real_Data_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Mounting Drive and importing Libraries**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
import os

# Verify GPU is active
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Running on CPU - training will be very slow!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu


#**Zip Extraction**

In [ ]:
import os
import zipfile

# Checking existence of zip in Drive
zip_path = '/content/drive/MyDrive/Thesis-work/ISIC_2019_Training_Input.zip'
print(f"ZIP exists: {os.path.exists(zip_path)}")

if os.path.exists(zip_path):
    # Creating extraction directory
    os.makedirs('data/isic2019', exist_ok=True)

    # Extraction
    print("Extracting... this may take 5-10 minutes")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data/isic2019/')

    print("Extraction complete")

    # Checking extraction content
    print("\nContents after extraction:")
    for item in os.listdir('data/isic2019/'):
        print(f"  {item}")

ZIP exists: True
Extracting... this may take 5-10 minutes
Extraction complete

Contents after extraction:
  ISIC_2019_Training_Input


**File Location**

In [ ]:
import os

# Checking what exists
print("Contents of current directory:")
for item in os.listdir('.'):
    print(f"  {item}")

if os.path.exists('data'):
    print("\nContents of data/:")
    for item in os.listdir('data'):
        print(f"  {item}")

    # Check nested folder
    nested = 'data/isic2019/ISIC_2019_Training_Input'
    if os.path.exists(nested):
        print(f"\nFound images in: {nested}")
        print(f"Image count: {len([f for f in os.listdir(nested) if f.endswith('.jpg')])}")

Contents of current directory:
  .config
  data
  drive
  sample_data

Contents of data/:
  isic2019

Found images in: data/isic2019/ISIC_2019_Training_Input
Image count: 25331


**Loading Data**

In [ ]:
backup_dir = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'

train_df = pd.read_csv(f'{backup_dir}/expA_train.csv')
val_df = pd.read_csv(f'{backup_dir}/expA_val.csv')
test_df = pd.read_csv(f'{backup_dir}/expA_test.csv')

IMAGE_DIR = 'data/isic2019/ISIC_2019_Training_Input'

class SkinDataset(Dataset):
  def __init__(self, df, transform=None):
    self.df = df
    self.transform = transform
    self.classes = sorted(df['label'].unique())
    self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img = Image.open(f"{IMAGE_DIR}/{row['image']}.jpg").convert('RGB')
    label = self.class_to_idx[row['label']]
    if self.transform:
      img = self.transform(img)
    return img, label


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.455, 0.406], [0.229, 0.224, 0.225])
])

train_ds = SkinDataset(train_df, transform)
val_ds = SkinDataset(val_df, transform)
test_ds = SkinDataset(test_df, transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Classes: {train_ds.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Train Distribution:", train_df['label'].value_counts().to_dict())

Classes: ['BKL', 'MEL', 'NV']
Train: 483, Val: 104, Test: 104
Train Distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}


**FOCAL LOSS-- better than class weights previously used for synthetic data**

In [ ]:
class FocalLoss(nn.Module):
  def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
    super().__init__()
    self.alpha = alpha ##Weighting Class
    self.gamma = gamma
    self.reduction = reduction

  def forward(self, inputs, targets):
    ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
    pt = torch.exp(-ce_loss)
    focal_term = (1 - pt) ** self.gamma
    loss = focal_term * ce_loss

    if self.reduction == 'mean':
      return loss.mean()
    elif self.reduction == 'sum':
      return loss.sum()
    return loss

##**Model Architecture**

In [ ]:
class SimpleEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten()
    )

  def forward(self, x):
    return self.features(x)

class SSLClassifier(nn.Module):
  def __init__(self, encoder, num_classes):
    super().__init__()
    self.encoder = encoder
    self.classifier = nn.Sequential(
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, num_classes)
    )

  def forward(self, x):
    return self.classifier(self.encoder(x))

#**Training with real Images**

**Oversampling dataloader**

In [ ]:
##Oversampling DataLoader
from torch.utils.data import WeightedRandomSampler

## Calculate weights for each sample (inverse frequency)
class_counts = train_df['label'].value_counts()
sample_weights = []
for label in train_df['label']:
    sample_weights.append(1.0 / class_counts[label])

# Standard DataLoader (no oversampling, just weights in loss)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Original train distribution: {dict(class_counts)}")
print(f"MEL weight: {1.0/class_counts['MEL']:.2f}")
print(f"NV weight: {1.0/class_counts['NV']:.2f}")
print("Oversampling DataLoader created")

Original train distribution: {'NV': np.int64(349), 'BKL': np.int64(99), 'MEL': np.int64(35)}
MEL weight: 0.03
NV weight: 0.00
Oversampling DataLoader created


**Load model + Stronger Weights**

In [ ]:
##Correct weight mapping based on actual class order

## Verifying class order
print(f"Class order: {train_ds.classes}")

## Create weights by CLASS NAME
weight_dict = {
    'MEL': 4.0,   # Rare, most important(The Goal)
    'BKL': 2.0,    # Medium
    'NV': 1.0      # Common
}

##Map to actual indices
weights_list = [weight_dict[cls] for cls in train_ds.classes]
weights = torch.tensor(weights_list, dtype=torch.float32).to(device)

print(f"Correct mapping: {dict(zip(train_ds.classes, weights_list))}")

##Loading model
encoder = SimpleEncoder()
ssl_weights = '/content/drive/MyDrive/Thesis-work/thesis_backup_day1/ssl_encoder_pretrained.pth'
encoder.load_state_dict(torch.load(ssl_weights))

for param in encoder.parameters():
    param.requires_grad = True

model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

##Setup loss and optimizer
criterion = FocalLoss(alpha=weights, gamma=1.5)
optimizer = optim.Adam(model.parameters(), lr=0.00005)

print("Model ready with CORRECTED weights")

Class order: ['BKL', 'MEL', 'NV']
Correct mapping: {'BKL': 2.0, 'MEL': 4.0, 'NV': 1.0}
Model ready with CORRECTED weights


**Training + Evaluation**

In [ ]:
## Training with oversampling
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

print("\nTraining with OVERSAMPLING...")
epochs = 40
best_val = 0
patience = 10
epochs_no_improve = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), 'best_oversampled.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

## Evaluation
model.load_state_dict(torch.load('best_oversampled.pth'))
test_acc, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*50)
print("RESULTS: OVERSAMPLING + STRONG WEIGHTS")
print("="*50)
print(f"Test Accuracy: {test_acc:.3f}")

from sklearn.metrics import classification_report
from collections import Counter
report = classification_report(true_labels, pred_labels, target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n>>> MEL Recall: {mel_recall:.1%} <<<")
print(f"Predictions: {Counter(pred_labels)}")

## Save to drive
torch.save(model.state_dict(), '/content/drive/MyDrive/Thesis-work/thesis_backup_day1/ssl_oversampled.pth')


Training with OVERSAMPLING...
Epoch 3: Train=0.675, Val=0.731
Epoch 6: Train=0.714, Val=0.712
Epoch 9: Train=0.706, Val=0.702
Early stopping at epoch 11

RESULTS: OVERSAMPLING + STRONG WEIGHTS
Test Accuracy: 0.692
              precision    recall  f1-score   support

         BKL       0.00      0.00      0.00        21
         MEL       0.00      0.00      0.00         8
          NV       0.71      0.96      0.82        75

    accuracy                           0.69       104
   macro avg       0.24      0.32      0.27       104
weighted avg       0.51      0.69      0.59       104


>>> MEL Recall: 0.0% <<<
Predictions: Counter({np.int64(2): 101, np.int64(0): 2, np.int64(1): 1})


**Diagnostic**

In [ ]:
# Check if images are actually loading correctly
sample_img, sample_label = train_ds[0]
print(f"Image shape: {sample_img.shape}")
print(f"Image value range: {sample_img.min():.3f} to {sample_img.max():.3f}")
print(f"Label: {sample_label} ({train_ds.classes[sample_label]})")

# Check if labels are correct
print("\nLabel distribution in first 20 samples:")
for i in range(20):
    _, label = train_ds[i]
    print(f"  Sample {i}: label={label} ({train_ds.classes[label]})")

# Check model output before training
model.eval()
with torch.no_grad():
    test_output = model(sample_img.unsqueeze(0).to(device))
    print(f"\nModel output (logits): {test_output}")
    print(f"Predicted class: {test_output.argmax(1).item()}")
    print(f"Softmax: {F.softmax(test_output, dim=1)}")

Image shape: torch.Size([3, 224, 224])
Image value range: -1.611 to 1.958
Label: 2 (NV)

Label distribution in first 20 samples:
  Sample 0: label=2 (NV)
  Sample 1: label=2 (NV)
  Sample 2: label=2 (NV)
  Sample 3: label=2 (NV)
  Sample 4: label=2 (NV)
  Sample 5: label=2 (NV)
  Sample 6: label=1 (MEL)
  Sample 7: label=2 (NV)
  Sample 8: label=2 (NV)
  Sample 9: label=2 (NV)
  Sample 10: label=2 (NV)
  Sample 11: label=2 (NV)
  Sample 12: label=1 (MEL)
  Sample 13: label=2 (NV)
  Sample 14: label=0 (BKL)
  Sample 15: label=2 (NV)
  Sample 16: label=0 (BKL)
  Sample 17: label=2 (NV)
  Sample 18: label=2 (NV)
  Sample 19: label=2 (NV)

Model output (logits): tensor([[ 0.5198, -0.4339, -0.1993]])
Predicted class: 0
Softmax: tensor([[0.5340, 0.2058, 0.2602]])
